📄 U-Net 구현 및 최적화 발표 자료 (README.md)
1. 프로젝트 개요
본 프로젝트는 ISBI 2012 EM segmentation 데이터를 활용하여 영상 내 세포 경계를 추출하는 U-Net 모델을 구현한 것입니다. 단순히 모델을 만드는 것에 그치지 않고, 대용량 데이터 처리를 위한 전처리 파이프라인 구축과 argparse를 이용한 실행 최적화에 초점을 맞췄습니다.

2. 주요 구현 특징
데이터 효율화: .tif 파일을 프레임별 .npy 파일로 쪼개어 메모리 부담을 줄이고 학습 속도를 향상시켰습니다.

유연한 아키텍처: 반복되는 Conv-BatchNorm-ReLU 구조를 CBR2d 함수로 모듈화하여 가독성을 높였습니다.

데이터 증강(Augmentation): 학습 데이터 부족 문제를 해결하기 위해 RandomFlip 등의 기법을 직접 정의하여 적용했습니다.

실험 관리: TensorBoard를 연동하여 Loss 변화와 실제 Segmentation 결과를 실시간으로 모니터링할 수 있도록 설계했습니다.

[Part 1] 데이터 전처리 및 로더 (dataset.py)
처음에는 단순히 이미지를 불러오는 방식에서, 나중에는 Transform을 통해 정규화와 텐서 변환을 자동화하는 방식으로 발전시켰습니다.

In [29]:
import os
import numpy as np
from PIL import Image

# 경로 설정
dir_data = './datasets'
name_label = 'train-labels.tif'
name_input = 'train-volume.tif'

# 폴더가 비어있는지 확인하고 데이터 생성
train_dir = os.path.join(dir_data, 'train')
if not os.path.exists(train_dir) or len(os.listdir(train_dir)) == 0:
    print("데이터를 생성합니다. 잠시만 기다려주세요...")
    
    img_label = Image.open(os.path.join(dir_data, name_label))
    img_input = Image.open(os.path.join(dir_data, name_input))

    nframe = img_label.n_frames
    id_frame = np.arange(nframe)
    np.random.shuffle(id_frame) # 데이터 셔플링

    # 하이퍼파라미터: 데이터셋 분할 비율
    nframe_train = 24
    nframe_val = 3
    nframe_test = 3

    # 데이터 저장 루프
    for mode, count, offset in [('train', nframe_train, 0), 
                                ('val', nframe_val, nframe_train), 
                                ('test', nframe_test, nframe_train + nframe_val)]:
        save_path = os.path.join(dir_data, mode)
        if not os.path.exists(save_path): os.makedirs(save_path)
        
        for i in range(count):
            idx = id_frame[i + offset]
            img_label.seek(idx)
            img_input.seek(idx)

            label_ = np.asarray(img_label)
            input_ = np.asarray(img_input)

            np.save(os.path.join(save_path, f'label_{i:03d}.npy'), label_)
            np.save(os.path.join(save_path, f'input_{i:03d}.npy'), input_)
    print(f"데이터 생성 완료: Train({nframe_train}), Val({nframe_val}), Test({nframe_test})")
else:
    print("데이터가 이미 존재합니다. 학습을 진행하세요.")

데이터를 생성합니다. 잠시만 기다려주세요...
데이터 생성 완료: Train(24), Val(3), Test(3)


In [22]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms # 4번 셀 에러 방지를 위해 미리 임포트

# [003, 005] Dataset 클래스 구현
class Dataset(torch.utils.data.Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform

        lst_data = os.listdir(self.data_dir)
        self.lst_label = sorted([f for f in lst_data if f.startswith('label')])
        self.lst_input = sorted([f for f in lst_data if f.startswith('input')])

    def __len__(self):
        return len(self.lst_label)

    def __getitem__(self, index):
        label = np.load(os.path.join(self.data_dir, self.lst_label[index]))
        input = np.load(os.path.join(self.data_dir, self.lst_input[index]))

        # 정규화: 0~255 범위를 0~1로 조정
        label, input = label / 255.0, input / 255.0

        if label.ndim == 2: label = label[:, :, np.newaxis]
        if input.ndim == 2: input = input[:, :, np.newaxis]

        data = {'input': input, 'label': label}
        if self.transform: data = self.transform(data)
        return data

# [005] Transform 구현: ToTensor, Normalization, RandomFlip
class ToTensor(object):
    def __call__(self, data):
        label, input = data['label'], data['input']
        # (H, W, C) -> (C, H, W) 축 변경
        label = label.transpose((2, 0, 1)).astype(np.float32)
        input = input.transpose((2, 0, 1)).astype(np.float32)
        return {'label': torch.from_numpy(label), 'input': torch.from_numpy(input)}

class Normalization(object):
    def __init__(self, mean=0.5, std=0.5):
        self.mean, self.std = mean, std
    def __call__(self, data):
        data['input'] = (data['input'] - self.mean) / self.std # 입력을 평균 0.5, 표준편차 0.5로 정규화
        return data

class RandomFlip(object):
    def __call__(self, data):
        if np.random.rand() > 0.5:
            data['label'] = np.fliplr(data['label']); data['input'] = np.fliplr(data['input'])
        if np.random.rand() > 0.5:
            data['label'] = np.flipud(data['label']); data['input'] = np.flipud(data['input'])
        return data

[Part 2] 네트워크 설계 (model.py)
U-Net의 핵심인 Skip Connection을 torch.cat으로 구현한 부분이 포인트입니다

In [30]:
class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()

        def CBR2d(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU()
            )

        # Encoder
        self.enc1_1 = CBR2d(1, 64); self.enc1_2 = CBR2d(64, 64); self.pool1 = nn.MaxPool2d(2)
        self.enc2_1 = CBR2d(64, 128); self.enc2_2 = CBR2d(128, 128); self.pool2 = nn.MaxPool2d(2)
        self.enc3_1 = CBR2d(128, 256); self.enc3_2 = CBR2d(256, 256); self.pool3 = nn.MaxPool2d(2)
        self.enc4_1 = CBR2d(256, 512); self.enc4_2 = CBR2d(512, 512); self.pool4 = nn.MaxPool2d(2)
        self.enc5_1 = CBR2d(512, 1024)

        # Decoder
        self.dec5_1 = CBR2d(1024, 512)
        self.unpool4 = nn.ConvTranspose2d(512, 512, kernel_size=2, stride=2)
        self.dec4_2 = CBR2d(1024, 512); self.dec4_1 = CBR2d(512, 256) # 512(up) + 512(skip)
        
        self.unpool3 = nn.ConvTranspose2d(256, 256, kernel_size=2, stride=2)
        self.dec3_2 = CBR2d(512, 256); self.dec3_1 = CBR2d(256, 128)

        self.unpool2 = nn.ConvTranspose2d(128, 128, kernel_size=2, stride=2)
        self.dec2_2 = CBR2d(256, 128); self.dec2_1 = CBR2d(128, 64)

        self.unpool1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec1_2 = CBR2d(128, 64); self.dec1_1 = CBR2d(64, 64)

        self.fc = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1_2(self.enc1_1(x)); p1 = self.pool1(e1)
        e2 = self.enc2_2(self.enc2_1(p1)); p2 = self.pool2(e2)
        e3 = self.enc3_2(self.enc3_1(p2)); p3 = self.pool3(e3)
        e4 = self.enc4_2(self.enc4_1(p3)); p4 = self.pool4(e4)
        e5 = self.enc5_1(p4)

        d5 = self.dec5_1(e5)
        # Skip Connection: dim=1(채널 방향)으로 Encoder 맵과 결합
        u4 = torch.cat((self.unpool4(d5), e4), dim=1)
        d4 = self.dec4_1(self.dec4_2(u4))

        u3 = torch.cat((self.unpool3(d4), e3), dim=1)
        d3 = self.dec3_1(self.dec3_2(u3))

        u2 = torch.cat((self.unpool2(d3), e2), dim=1)
        d2 = self.dec2_1(self.dec2_2(u2))

        u1 = torch.cat((self.unpool1(d2), e1), dim=1)
        d1 = self.dec1_1(self.dec1_2(u1))

        return self.fc(d1)

[Part 3] 메인 학습 루프 및 최적화 (train.py)
강의 008의 핵심인 argparse를 적용하여 코드를 유연하게 만들었습니다.

In [31]:
import argparse

# 모델 저장 및 로드 함수
def save(ckpt_dir, net, optim, epoch):
    if not os.path.exists(ckpt_dir): os.makedirs(ckpt_dir)
    torch.save({'net': net.state_dict(), 'optim': optim.state_dict()}, f"{ckpt_dir}/model_epoch{epoch}.pth")

def load(ckpt_dir, net, optim):
    if not os.path.exists(ckpt_dir): return net, optim, 0
    ckpt_lst = sorted(os.listdir(ckpt_dir), key=lambda f: int(''.join(filter(str.isdigit, f))))
    dict_model = torch.load(f"{ckpt_dir}/{ckpt_lst[-1]}")
    net.load_state_dict(dict_model['net']); optim.load_state_dict(dict_model['optim'])
    return net, optim, int(ckpt_lst[-1].split('epoch')[1].split('.pth')[0])

# 파라미터 설정
parser = argparse.ArgumentParser(description="UNet Training", formatter_class=argparse.ArgumentDefaultsHelpFormatter)
parser.add_argument("--lr", default=1e-3, type=float, dest="lr")
parser.add_argument("--batch_size", default=4, type=int, dest="batch_size")
parser.add_argument("--num_epoch", default=100, type=int, dest="num_epoch")
parser.add_argument("--data_dir", default="./datasets", type=str, dest="data_dir")
parser.add_argument("--ckpt_dir", default="./checkpoint", type=str, dest="ckpt_dir")
parser.add_argument("--mode", default="train", type=str, dest="mode")

args = parser.parse_args(args=[]) # 노트북 에러 방지

In [ ]:
# 4번 셀: 학습 루프 실행
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = UNet().to(device)
fn_loss = nn.BCEWithLogitsLoss().to(device)
optim = torch.optim.Adam(net.parameters(), lr=args.lr)

# 경로 확인 로직 추가
if not os.path.exists(os.path.join(args.data_dir, 'train')):
    print(f"Error: {args.data_dir}/train 경로에 데이터가 없습니다. 0번 셀을 먼저 확인하세요.")
else:
    if args.mode == "train":
        transform = transforms.Compose([Normalization(), RandomFlip(), ToTensor()])
        dataset_train = Dataset(data_dir=os.path.join(args.data_dir, 'train'), transform=transform)
        loader_train = DataLoader(dataset_train, batch_size=args.batch_size, shuffle=True)

        print(f"국민대 X_AI 학회 U-Net 학습 시작 (Device: {device})")
        
        for epoch in range(1, args.num_epoch + 1):
            net.train()
            loss_arr = []

            for batch, data in enumerate(loader_train, 1):
                label = data['label'].to(device)
                input = data['input'].to(device)

                output = net(input)

                optim.zero_grad()
                loss = fn_loss(output, label)
                loss.backward()
                optim.step()

                loss_arr.append(loss.item())

            # 10에폭마다 결과 출력
            if epoch % 10 == 0 or epoch == 1:
                print(f"Epoch {epoch:03d}/{args.num_epoch} | Loss: {np.mean(loss_arr):.4f}")
            
            # 모델 저장
            if epoch % 50 == 0:
                save(args.ckpt_dir, net, optim, epoch)

국민대 X_AI 학회 U-Net 학습 시작 (Device: cpu)
Epoch 001/100 | Loss: 0.5543


전처리 (003, 005): "저희가 처음에 큰 .tif 파일을 한꺼번에 올리려고 하니 메모리 문제가 있었습니다. 그래서 강의에서 배운 대로 각 프레임을 .npy로 쪼개서 저장하는 방식을 채택했고, Dataset 클래스에서 실시간으로 불러오게 구현하여 효율성을 높였습니다."

모델 구조 (004): "U-Net의 핵심은 인코더의 특징을 디코더로 직접 전달하는 Skip Connection입니다. 코드에서는 torch.cat 함수를 통해 채널 방향으로 결합해 주었습니다. 또한, 단순한 업샘플링 대신 학습 가능한 ConvTranspose2d를 사용하여 복원력을 높였습니다."

최적화 (008): "마지막으로 코드의 범용성을 위해 모든 하이퍼파라미터를 argparse로 관리하도록 수정했습니다. 덕분에 터미널에서 코드 수정 없이 다양한 실험(Learning Rate 변경 등)을 빠르게 진행할 수 있었습니다."